# Import Contracts, Combine & Next Gen Stats to Bronze

Run this notebook in Microsoft Fabric with your target Lakehouse attached.

Prerequisites:

- Run `scripts/acquire_contracts_combine_nextgen.py` locally to download the Parquet files.
- Upload the following folders from `nflverse_local/raw/nflverse/` to Lakehouse `Files/raw/nflverse/`:
  - `contracts/`
  - `combine/`
  - `nextgen_stats/`
- Use a schema-enabled Lakehouse. This notebook writes to the existing `bronze` schema.

The notebook writes managed Delta tables under the `bronze` schema while preserving source columns and adding ingestion metadata.

In [ ]:
from datetime import datetime, timezone

from pyspark.sql import DataFrame
from functools import reduce

from pyspark.sql.functions import col, current_timestamp, input_file_name, lit
from pyspark.sql.types import (
    BooleanType,
    ByteType,
    DateType,
    DecimalType,
    DoubleType,
    FloatType,
    IntegerType,
    LongType,
    ShortType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

RAW_ROOT = "Files/raw/nflverse"
BRONZE_SCHEMA = "bronze"
WRITE_MODE = "overwrite"

ACQUISITION_BATCH_ID = "nflverse_supplemental_contracts_combine_nextgen"
NOTEBOOK_RUN_UTC = datetime.now(timezone.utc).isoformat()

print(f"Notebook run UTC: {NOTEBOOK_RUN_UTC}")
print(f"Raw root: {RAW_ROOT}")
print(f"Bronze schema: {BRONZE_SCHEMA}")

## Ensure Bronze Schema Exists

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{BRONZE_SCHEMA}`")
spark.sql(f"SHOW TABLES IN `{BRONZE_SCHEMA}`").show(200, truncate=False)

## Source-To-Table Map

| Raw Folder | Bronze Table | Description |
|---|---|---|
| `contracts/` | `nflverse_contracts` | Player contract data from OverTheCap (salary, cap hit, guarantees). |
| `combine/` | `nflverse_combine` | NFL Combine athletic measurement results. |
| `nextgen_stats/` | `nflverse_nextgen_stats` | Next Gen Stats advanced metrics (passing, rushing, receiving). |

In [ ]:
TABLE_SPECS = [
    {
        "table": "nflverse_contracts",
        "source": "contracts",
        "description": "Raw nflverse player contract data from OverTheCap (salary, cap hit, guarantees, signing bonus).",
    },
    {
        "table": "nflverse_combine",
        "source": "combine",
        "description": "Raw nflverse NFL Combine athletic measurement results (40-yard dash, bench press, vertical, etc.).",
    },
    {
        "table": "nflverse_nextgen_stats",
        "source": "nextgen_stats",
        "description": "Raw nflverse Next Gen Stats advanced metrics for passing, rushing, and receiving.",
    },
]

for spec in TABLE_SPECS:
    print(f"{BRONZE_SCHEMA}.{spec['table']} <- {RAW_ROOT}/{spec['source']}")

## Import Helpers

These helpers handle schema drift across Parquet files by reading each file individually, computing a compatible schema, and unioning the results.

In [ ]:
def log(message: str) -> None:
    print(message, flush=True)


def full_table_name(table_name: str) -> str:
    return f"{BRONZE_SCHEMA}.{table_name}"


def quoted_table_name(table_name: str) -> str:
    return f"`{BRONZE_SCHEMA}`.`{table_name}`"


def sql_string(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"


def source_path(spec: dict) -> str:
    return f"{RAW_ROOT}/{spec['source']}"


def file_info_is_dir(file_info) -> bool:
    if hasattr(file_info, "isDir"):
        return bool(file_info.isDir)
    if hasattr(file_info, "is_dir"):
        return bool(file_info.is_dir)
    item_type = str(getattr(file_info, "type", "")).lower()
    return item_type in {"directory", "dir"} or str(file_info.path).endswith("/")


def ls_path(path: str):
    if "notebookutils" in globals():
        return notebookutils.fs.ls(path)
    if "mssparkutils" in globals():
        return mssparkutils.fs.ls(path)
    raise RuntimeError("Fabric notebook file utilities are unavailable in this session.")


def list_parquet_files(path: str) -> list[str]:
    files = []
    for item in ls_path(path):
        item_path = item.path
        if file_info_is_dir(item):
            files.extend(list_parquet_files(item_path))
        elif item_path.lower().endswith(".parquet"):
            files.append(item_path)
    return sorted(files)


def choose_common_type(data_types):
    unique_types = {type(data_type) for data_type in data_types}

    if len(unique_types) == 1:
        return data_types[0]

    if StringType in unique_types:
        return StringType()

    numeric_types = {ByteType, ShortType, IntegerType, LongType, FloatType, DoubleType, DecimalType}
    if unique_types.issubset(numeric_types):
        if unique_types.intersection({FloatType, DoubleType, DecimalType}):
            return DoubleType()
        return LongType()

    if unique_types.issubset({DateType, TimestampType}):
        return TimestampType()

    if unique_types == {BooleanType, IntegerType}:
        return LongType()

    return StringType()


def build_common_schema(files: list[str]) -> StructType:
    field_order = []
    types_by_field = {}

    for file_path in files:
        schema = spark.read.parquet(file_path).schema
        for field in schema.fields:
            if field.name not in types_by_field:
                field_order.append(field.name)
                types_by_field[field.name] = []
            types_by_field[field.name].append(field.dataType)

    return StructType(
        [
            StructField(field_name, choose_common_type(types_by_field[field_name]), True)
            for field_name in field_order
        ]
    )


def conform_to_schema(df: DataFrame, target_schema: StructType) -> DataFrame:
    existing_columns = set(df.columns)

    for field in target_schema.fields:
        if field.name in existing_columns:
            df = df.withColumn(field.name, col(field.name).cast(field.dataType))
        else:
            df = df.withColumn(field.name, lit(None).cast(field.dataType))

    return df.select([col(field.name) for field in target_schema.fields])


def read_raw_parquet(path: str) -> DataFrame:
    files = list_parquet_files(path)
    if not files:
        raise FileNotFoundError(f"No Parquet files found under {path}")

    log(f"Building compatible schema from {len(files)} Parquet file(s) under {path}")
    target_schema = build_common_schema(files)
    log(f"Found {len(files)} Parquet file(s); using compatible schema with {len(target_schema.fields)} columns")

    dataframes = [
        conform_to_schema(spark.read.parquet(file_path), target_schema)
        .withColumn("_source_file_path", lit(file_path))
        for file_path in files
    ]

    return reduce(lambda left, right: left.unionByName(right), dataframes)


def add_bronze_metadata(df: DataFrame, spec: dict) -> DataFrame:
    return (
        df.withColumn("_bronze_ingested_at_utc", current_timestamp())
        .withColumn("_source_file_path", col("_source_file_path") if "_source_file_path" in df.columns else input_file_name())
        .withColumn("_source_system", lit("nflreadpy"))
        .withColumn("_source_dataset", lit(spec["table"]))
        .withColumn("_acquisition_batch_id", lit(ACQUISITION_BATCH_ID))
    )


def write_bronze_table(spec: dict) -> dict:
    path = source_path(spec)
    table_name = full_table_name(spec["table"])
    quoted_name = quoted_table_name(spec["table"])

    log(f"Reading {path}")
    df = add_bronze_metadata(read_raw_parquet(path), spec)
    source_columns = len(df.columns)
    source_rows = df.count()

    log(f"Writing {table_name}: {source_rows:,} rows, {source_columns:,} columns")
    (
        df.write
        .format("delta")
        .mode(WRITE_MODE)
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    log(f"Verifying {table_name} row count")
    table_rows = spark.table(table_name).count()
    spark.sql(f"COMMENT ON TABLE {quoted_name} IS {sql_string(spec['description'])}")
    log(f"Finished {table_name}: {table_rows:,} rows")

    return {
        "schema": BRONZE_SCHEMA,
        "table": spec["table"],
        "source_path": path,
        "source_rows": source_rows,
        "table_rows": table_rows,
        "columns": source_columns,
        "row_count_matches": source_rows == table_rows,
    }

## Write Bronze Tables

This cell overwrites the supplemental Bronze tables. Re-running is safe and idempotent.

In [ ]:
results = []
for spec in TABLE_SPECS:
    result = write_bronze_table(spec)
    results.append(result)
    print()

## Summary

In [ ]:
import pandas as pd

summary_df = pd.DataFrame(results)
display(summary_df)

mismatches = summary_df[~summary_df["row_count_matches"]]
if len(mismatches) > 0:
    print(f"\n⚠️  {len(mismatches)} table(s) have row count mismatches:")
    display(mismatches)
else:
    print(f"\n✅ All {len(results)} table(s) imported successfully with matching row counts.")